In [0]:
from pyspark.sql.functions import col, countDistinct, sum as spark_sum, when

def build_adls_options(storage_account_name, client_id, tenant_id, client_secret):
    return {
        f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net": "OAuth",
        f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
        f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net": client_id,
        f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net": client_secret,
        f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
    }


def validate_required_columns(df, expected_columns):
    missing_columns = [c for c in expected_columns if c not in df.columns]
    unexpected_columns = [c for c in df.columns if c not in expected_columns]

    assert not missing_columns, f"Colunas esperadas ausentes: {missing_columns}"

    return unexpected_columns


def get_null_summary(df):
    return df.select([
        spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df.columns
    ])


def get_basic_dq_metrics(df):
    total_linhas = df.count()
    total_colunas = len(df.columns)

    return {
        "total_linhas": total_linhas,
        "total_colunas": total_colunas,
        "id_cliente_nulos": df.filter(col("id_cliente").isNull()).count() if "id_cliente" in df.columns else None,
        "id_cliente_distintos": df.select(countDistinct("id_cliente")).collect()[0][0] if "id_cliente" in df.columns else None,
        "uuid_cliente_nulos": df.filter(col("uuid_cliente").isNull()).count() if "uuid_cliente" in df.columns else None,
        "uuid_cliente_distintos": df.select(countDistinct("uuid_cliente")).collect()[0][0] if "uuid_cliente" in df.columns else None,
    }


def read_sql_table(spark, sql_host, sql_database, sql_username, sql_password, table_name):
    return (
        spark.read
        .format("sqlserver")
        .option("host", sql_host)
        .option("port", "1433")
        .option("database", sql_database)
        .option("dbtable", table_name)
        .option("user", sql_username)
        .option("password", sql_password)
        .load()
    )


def write_sql_table(df, sql_host, sql_database, sql_username, sql_password, table_name, mode="overwrite"):
    (
        df.write
        .format("sqlserver")
        .option("host", sql_host)
        .option("port", "1433")
        .option("database", sql_database)
        .option("dbtable", table_name)
        .option("user", sql_username)
        .option("password", sql_password)
        .mode(mode)
        .save()
    )